# 2. Agent

This notebook shows a simple LangChain agent with tools and memory.

In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

## Defining tools

A tool has two especially important parts:

- the input schema, which tells LangChain what arguments the tool expects
- the description, usually the function docstring, which helps the agent understand when the tool is useful

Here we use Pydantic models to define the tool inputs explicitly.

In [3]:
class WeatherInput(BaseModel):
    city: str = Field(description="The city for which we want the weather.")


class FavoriteFoodInput(BaseModel):
    animal: str = Field(description="The animal for which we want the favorite food.")

In [4]:
@tool(args_schema=WeatherInput)
def get_weather(city: str) -> str:
    """Use this tool when the user asks for the weather in a city."""
    fake_weather = {
        "brussels": "Cloudy, 18C",
        "paris": "Sunny, 24C",
        "london": "Rainy, 16C",
    }
    return fake_weather.get(city.lower(), f"No fake weather data found for {city}.")

In [5]:
@tool(args_schema=FavoriteFoodInput)
def get_favorite_food(animal: str) -> str:
    """Use this tool when the user asks what food a given animal likes."""
    fake_foods = {
        "cat": "salmon",
        "dog": "chicken",
        "rabbit": "carrots",
    }
    return fake_foods.get(animal.lower(), f"I do not know the favorite food of {animal}.")

LangChain uses the Pydantic schema to know the tool arguments and their field descriptions.

The agent also reads the tool description from the docstring, which is why it is worth writing it clearly: it helps the model decide which tool to call and when.

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_weather, get_favorite_food],
    system_prompt="You are a helpful assistant. Use the available tools when they help.",
    checkpointer=memory,
)

In [7]:
config = {"configurable": {"thread_id": "tutorial-thread"}}

In [8]:
first_response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What is the weather in Brussels and what is the favorite food "
                    "of a cat? Answer in 2 short bullet points."
                ),
            }
        ]
    },
    config=config,
)

print("First answer:")
print(first_response["messages"][-1].content)

First answer:
[{'type': 'text', 'text': '* The weather in Brussels is cloudy with a temperature of 18°C.\n* The favorite food of a cat is salmon.', 'extras': {'signature': 'EjQKMgEMOdbHsisQQwJf7kKN1MY3XEfnRdmvAldKJZUMgiLxojqH1Ea4GEu7e2WzcbzpU6lE'}}]


In [9]:
second_response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Now only remind me of the favorite food from the animal I asked "
                    "about earlier. Answer in one short sentence."
                ),
            }
        ]
    },
    config=config,
)

print("Second answer:")
print(second_response["messages"][-1].content)

Second answer:
[{'type': 'text', 'text': 'The favorite food of a cat is salmon.', 'extras': {'signature': 'EjQKMgEMOdbHhtmD9NJ8MxZxl6LR++kXXpjR2Ewo0yBNkd1e0ruzm1yQZJOaWt6HJi4nJGc2'}}]


## Streaming the agent execution

We can also stream the agent run to observe when tools are called and how the final answer is produced token by token.

In [11]:
stream_prompt = {
    "messages": [
        {
            "role": "user",
            "content": (
                "What is the weather in Paris and what is the favorite food of a rabbit? "
                "Answer in 2 short bullet points."
            ),
        }
    ]
}

print("Streaming run:\n")

for chunk in agent.stream(stream_prompt, config=config, stream_mode="values"):
    message = chunk["messages"][-1]

    if getattr(message, "tool_calls", None):
        print("Tool calls:")
        for tool_call in message.tool_calls:
            print(tool_call)
        print()

    text = message.text
    if text:
        print(text)


Streaming run:

What is the weather in Paris and what is the favorite food of a rabbit? Answer in 2 short bullet points.
* The weather in Paris is sunny with a temperature of 24°C.
* The favorite food of a rabbit is carrots.
